# Volume 4, Chapter 3: Classic Series and Infinite Products (Python companion)

Exercises first, solutions at the end. We check Euler's formula for $\zeta(2k)$ to fifty digits, race the products of Wallis and Viète, rediscover identities from their decimals with the PSLQ algorithm, accelerate the slowest series of all, measure the error of the sine product, and test the theorem on the period of $\sqrt d$ far beyond hand computation.

*Before each solution, predict what the output will look like.*

## Exercises

**P3.1 (★) Fifty digits.** Check Theorem 3.7 to $50$ digits for $k=1,\dots,10$ with mpmath. Then compare the speed of Wallis's and Viète's products (Corollary 3.6): how many correct digits per factor does each give?

**P3.2 (★★) Experimental mathematics: PSLQ.** Given real numbers $x_1,\dots,x_n$, the PSLQ algorithm (Ferguson and Bailey, 1992) searches for integers $c_i$, not all zero, with $\sum c_ix_i=0$. Use it to rediscover $\zeta(4)=\frac{\pi^4}{90}$ from $40$ digits, to look (in vain) for $\zeta(3)$ as a rational multiple of $\pi^3$, and to find Machin's formula relating $\pi$, $\arctan\frac15$ and $\arctan\frac1{239}$.

**P3.3 (★★) CS application: convergence acceleration.** Leibniz's series (Exercise 3.1) gives about one digit per ten terms. Compare, after $20$ terms, the plain partial sums, Aitken's $\Delta^2$ process $s_n-\frac{(\Delta s_n)^2}{\Delta^2s_n}$ (applied once and three times), and Euler's transform $\sum_k(-1)^ka_k=\sum_n(-1)^n\frac{\Delta^na_0}{2^{n+1}}$, with $\Delta$ the difference operator of Chapter 1.

**P3.4 (★) How fast is Euler's product?** Measure the relative error of $\pi x\prod_{n\le N}\bigl(1-\frac{x^2}{n^2}\bigr)$ against $\sin\pi x$ (Theorem 3.5) and compare with $\frac{x^2}N$. Check Theorem 3.4 numerically too.

**P3.5 (★★) Periods of $\sqrt d$.** Compute the continued fraction of $\sqrt d$ with exact integer arithmetic for every non-square $d<10\,000$, and check Exercise 3.6: the period ends with $2a_0$. Is the rest of the period a palindrome? Which $d$ has the longest period?

**P3.6 (★★) The size of Bernoulli numbers.** Compare $|B_{2k}|$ with $\frac{2(2k)!}{(2\pi)^{2k}}$ for $k$ up to $40$. What is the ratio exactly?

## Solutions

### P3.1

In [ ]:
import mpmath as mp
mp.mp.dps = 50
for k in range(1, 11):
    lhs = mp.zeta(2 * k)
    rhs = (-1)**(k + 1) * mp.bernoulli(2 * k) * (2 * mp.pi)**(2 * k) / (2 * mp.factorial(2 * k))
    print(f'k = {k:2d}   zeta(2k) = {mp.nstr(lhs, 20)}   |difference| = {mp.nstr(abs(lhs - rhs), 3)}')

In [ ]:
def wallis(N):
    return mp.fprod((2 * n)**2 / mp.mpf((2 * n - 1) * (2 * n + 1)) for n in range(1, N + 1))
def viete(K):
    c, p = mp.sqrt(2) / 2, mp.mpf(1)
    for _ in range(K):
        p *= c; c = mp.sqrt((1 + c) / 2)
    return 2 / p
print(' N     Wallis error      K   Viete error')
for N, K in [(10, 5), (100, 10), (1000, 20), (10000, 40)]:
    print(f'{N:6d}  {mp.nstr(mp.pi / 2 - wallis(N), 3):>12}    {K:3d}   {mp.nstr(mp.pi - viete(K), 3):>10}')

The formula holds to the last of the $50$ digits (the differences are rounding errors). Wallis's product converges like $\frac\pi{8N}$: one more digit costs ten times more factors. Viète's error is divided by $4$ at each factor, so each factor gives $\log_{10}4\approx0.6$ digits: the first algorithm in history with *linear* convergence. Chapter 12 will show an algorithm whose number of correct digits doubles at each step.

### P3.2

In [ ]:
mp.mp.dps = 40
print('PSLQ on (zeta(4), pi^4):', mp.pslq([mp.zeta(4), mp.pi**4], maxcoeff=10**6, maxsteps=10**6))
print('PSLQ on (zeta(3), pi^3):', mp.pslq([mp.zeta(3), mp.pi**3], maxcoeff=10**6, maxsteps=10**6))
print('PSLQ on (pi, atan 1/5, atan 1/239):', mp.pslq([mp.pi, mp.atan(mp.mpf(1) / 5), mp.atan(mp.mpf(1) / 239)], maxcoeff=1000))
print('identify(zeta(6)/pi^6):', mp.identify(mp.zeta(6) / mp.pi**6))

PSLQ finds $-90\,\zeta(4)+\pi^4=0$ and Machin's $\pi-16\arctan\frac15+4\arctan\frac1{239}=0$, that is, $\frac\pi4=4\arctan\frac15-\arctan\frac1{239}$, the formula Machin used in 1706 to compute $100$ digits of $\pi$. For $\zeta(3)$ and $\pi^3$ it finds no relation with coefficients below $10^6$, which is evidence, not proof: such searches have ruled out small relations between $\zeta(3)$ and $\pi^3$ with coefficients up to astronomical bounds. This is how many identities are discovered today: compute to high precision, search for a relation, then prove it.

### P3.3

In [ ]:
mp.mp.dps = 30
terms = [mp.mpf((-1)**k) / (2 * k + 1) for k in range(40)]
partial = [mp.fsum(terms[:n + 1]) for n in range(40)]
target = mp.pi / 4
def aitken(s):
    return [s[n + 2] - (s[n + 2] - s[n + 1])**2 / ((s[n + 2] - s[n + 1]) - (s[n + 1] - s[n])) for n in range(len(s) - 2)]
def euler_transform(N):
    # sum_k (-1)^k a_k = sum_n (-1)^n Delta^n a_0 / 2^(n+1), with Delta the forward difference
    a = [mp.mpf(1) / (2 * k + 1) for k in range(N + 1)]
    out, row = mp.mpf(0), a
    for n in range(N):
        out += (-1)**n * row[0] / 2**(n + 1)
        row = [row[i + 1] - row[i] for i in range(len(row) - 1)]
    return out
A1 = aitken(partial); A2 = aitken(A1); A3 = aitken(A2)
digits = lambda x: float(-mp.log10(abs(x - target)))
print('correct digits after 20 terms:  plain', round(digits(partial[19]), 1), '  Aitken once', round(digits(A1[17]), 1),
      '  Aitken three times', round(digits(A3[13]), 1), '  Euler transform', round(digits(euler_transform(20)), 1))
print('mpmath nsum (Levin-type acceleration):', mp.nstr(mp.nsum(lambda k: (-1)**k / (2 * k + 1), [0, mp.inf]) - target, 3))

After $20$ terms the plain sum has about $2$ correct digits, Aitken's process $5$, three Aitken passes about $10$, and Euler's transform about $7$; mpmath's `nsum`, which uses Levin-type and Richardson extrapolation, reaches full precision. The reason acceleration works here is the smooth structure of the error: $\frac\pi4-s_n$ has an asymptotic expansion in powers of $\frac1n$ with alternating sign, which these transforms cancel term after term. On a series with random-looking terms they would do nothing.

### P3.4

In [ ]:
mp.mp.dps = 20
def sine_product(x, N):
    return mp.pi * x * mp.fprod(1 - x**2 / mp.mpf(n)**2 for n in range(1, N + 1))
print('   x      N    relative error    x^2/N')
for x in (0.3, 1.5, 2.5):
    for N in (10, 100, 1000):
        err = abs(sine_product(x, N) / mp.sin(mp.pi * x) - 1)
        print(f'{x:5.1f}  {N:5d}    {mp.nstr(err, 4):>12}    {x * x / N:.4g}')
x = mp.mpf('0.3')
print('cot partial fractions, 10^5 terms, x = 0.3:', mp.nstr(mp.pi * mp.cot(mp.pi * x) - (1 / x + mp.fsum(2 * x / (x**2 - n**2) for n in range(1, 100001))), 3))

The relative error is close to $\frac{x^2}N$, as the tail $\prod_{n>N}\bigl(1-\frac{x^2}{n^2}\bigr)\approx\exp\bigl(-\sum_{n>N}\frac{x^2}{n^2}\bigr)\approx1-\frac{x^2}N$ predicts, and the agreement improves as $N$ grows. Theorem 3.4 with $10^5$ terms is off by about $-\frac{2x}N=-6\times10^{-6}$, the size of the neglected tail: product and partial fractions converge like $\frac1N$, which is why one uses them for theory and other formulas for computation.

### P3.5

In [ ]:
from math import isqrt
def cf_sqrt(d):
    a0 = isqrt(d); m, q, a = 0, 1, a0; period = []
    while True:
        m = q * a - m; q = (d - m * m) // q; a = (a0 + m) // q
        period.append(a)
        if a == 2 * a0: return a0, period
ok_end, ok_pal, longest = True, True, (0, 0)
for d in range(2, 10000):
    if isqrt(d)**2 == d: continue
    a0, per = cf_sqrt(d)
    ok_end &= per[-1] == 2 * a0
    ok_pal &= per[:-1] == per[:-1][::-1]
    longest = max(longest, (len(per), d))
print('period ends with 2a_0 for all d < 10000:', ok_end, '  and a_1..a_(l-1) is a palindrome:', ok_pal)
print('longest period below 10000: length', longest[0], 'for d =', longest[1])
print('sqrt(14):', cf_sqrt(14), '  sqrt(61):', cf_sqrt(61))

The period always ends with $2a_0$, and the part before it, $a_1,\dots,a_{\ell-1}$, is always a palindrome (also a classical theorem, proved by the same conjugation argument). Periods can be long: $\sqrt{9949}$ has period $217$. For $d=61$ the period has length $11$, odd, so the convergent at the end of the first period solves $x^2-61y^2=-1$, and one must go to the end of the second period to solve the Pell equation $x^2-61y^2=1$ (Volume 1, Chapter 8): its fundamental solution is $x=1\,766\,319\,049$, $y=226\,153\,980$.

### P3.6

In [ ]:
mp.mp.dps = 30
for k in (1, 2, 5, 10, 20, 40):
    B = mp.bernoulli(2 * k)
    approx = 2 * mp.factorial(2 * k) / (2 * mp.pi)**(2 * k)
    print(f'k = {k:2d}   B_2k = {mp.nstr(B, 8):>16}   |B_2k| / (2(2k)!/(2pi)^2k) = {mp.nstr(abs(B) / approx, 12)}')

The ratio is exactly $\zeta(2k)$, by Theorem 3.7, and it tends to $1$ very fast. So $|B_{2k}|\approx\frac{2(2k)!}{(2\pi)^{2k}}$ with a relative error about $2^{-2k}$: $B_{80}\approx-2.0\times10^{55}$. The Bernoulli numbers start small and then grow faster than any exponential, which Chapter 11 will meet again in the divergence of the Euler–Maclaurin and Stirling series.